# Lab 03 · Phân cụm khách hàng

So sánh K-means và Gaussian Mixture trên Mall Customers. Xem công thức, lý do chọn chỉ số và phần diễn giải trong `docs/Lab_03_descript.md`.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

root = Path.cwd()
if not (root / "config.yaml").is_file():
    root = root.parent
if not (root / "config.yaml").is_file():
    raise FileNotFoundError("Mở notebook từ thư mục LA03 hoặc notebooks.")
sys.path.insert(0, str(root))

from src.pipeline import run_pipeline

run = run_pipeline(root / "config.yaml")
reports = root / run["paths"]["reports"]
images = root / run["paths"]["images"]
print(run["run_id"])

## Khám phá dữ liệu

Xem số giá trị thiếu, các điểm bị đánh dấu bởi quy tắc IQR và quan hệ giữa từng cặp biến.

In [ ]:
eda = json.loads((reports / "eda.json").read_text(encoding="utf-8"))
display(pd.Series(eda["missing_raw"], name="Giá trị thiếu"))
display(pd.DataFrame(eda["iqr_flagged_rows"]))
display(Image(filename=str(images / "eda.png"), width=950))

## Chọn cấu hình trên tập train

Silhouette trung bình cao nhất là tiêu chí chọn cấu hình. Davies–Bouldin và Calinski–Harabasz giúp xem lựa chọn thay đổi ra sao khi đánh giá theo độ gọn và khoảng cách giữa các tâm.

In [ ]:
columns = ["family", "k", "silhouette_mean", "silhouette_std",
           "davies_bouldin_mean", "calinski_harabasz_mean", "eligible"]
display(pd.read_csv(reports / "cv_summary.csv")[columns].round(4))
display(Image(filename=str(images / "cv_metrics.png"), width=1000))

## Đọc kết quả trên train và test

Dùng hồ sơ trên thang đo gốc để mô tả cụm. Biểu đồ Silhouette cho thấy điểm nào gần cụm khác theo khoảng cách trung bình.

In [ ]:
metrics = json.loads((reports / "metrics.json").read_text(encoding="utf-8"))
display(pd.DataFrame([metrics["train"], metrics["test"]], index=["train", "test"]).round(4))
display(pd.Series(metrics["stability"], name="ARI"))
display(pd.read_csv(reports / "train_profile.csv").round(2))
display(pd.read_csv(reports / "test_profile.csv").round(2))
display(Image(filename=str(images / "segments.png"), width=950))
display(Image(filename=str(images / "silhouette.png"), width=950))

## Gán cụm cho khách mới

Ví dụ dùng lại bộ chuẩn hóa và mô hình đã lưu từ lần chạy trên.

In [ ]:
from src.model import load_model
from src.predict import assign_customer

bundle = load_model(root / run["paths"]["model"] / "customer_clusters.joblib")
assign_customer(bundle, age=35, income=70, spending=50)

## Bài bổ sung · Phân đoạn ảnh theo màu

Chạy K-means với 3, 6 và 9 cụm màu trên ảnh mẫu Grace Hopper.

In [ ]:
from src.image_segmentation import segment_image

display(segment_image().round(2))
display(Image(filename=str(root / "docs/images/image_segmentation/comparison.png"), width=950))